# Chunking Retrieval Debugger

Inspect why specific questions fail for different chunking strategies.
Ask a question, see the top-k chunks retrieved by each chunker, and compare.

In [11]:
import sys
from pathlib import Path

# Setup paths — works whether you run from experiments/chunking/ or project root
if Path.cwd().name == 'chunking':
    PROJECT_ROOT = Path.cwd().parent.parent
else:
    PROJECT_ROOT = Path.cwd()

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'parsers'))

import numpy as np
import json
import pymupdf4llm

from experiments.chunking.chunkers import (
    get_all_chunkers, split_into_pages, split_into_elements,
    count_tokens, Chunk, ChunkType,
    PageLevelChunker, FixedSizeTokenChunker, RecursiveChunker,
    HeadingBasedChunker, TableAwareChunker, ElementTypeChunker,
    SemanticChunker, StructureAwareChunker,
)
from experiments.chunking.embedder import Embedder, cosine_similarity, BM25Index, hybrid_scores
from pymupdf_fast import parse_document as pymupdf_fast_parse

# Load QA pairs
with open(PROJECT_ROOT / 'experiments' / 'chunking' / 'qa_pairs.json') as f:
    qa_data = json.load(f)['documents']

embedder = Embedder()
print('Imports OK')
print(f'Project root: {PROJECT_ROOT}')
print(f'Available docs: {list(qa_data.keys())}')

Imports OK
Project root: /Users/sarath/work/doc-parser
Available docs: ['VAM-3852AO', 'hubspot-q4', 'hubspot-deck', '2023_report_40_pages', '10Q', 'kiski', 'report']


## 1. Load a document with both parsers

In [12]:
# ---- EDIT THIS ----
DOC = 'hubspot-q4'
# -------------------

pdf_path = PROJECT_ROOT / 'docs' / f'{DOC}.pdf'

# Parse with both approaches
md_fast = pymupdf_fast_parse(pdf_path)
md_4llm = pymupdf4llm.to_markdown(str(pdf_path), show_progress=False)

# Load questions
questions = qa_data[DOC]['questions']

print(f'Document: {DOC}')
print(f'pymupdf_fast:  {len(md_fast):>8} chars, {len(md_fast.splitlines()):>5} lines')
print(f'pymupdf4llm:   {len(md_4llm):>8} chars, {len(md_4llm.splitlines()):>5} lines')

# Count pipe-table lines
pipes_fast = sum(1 for l in md_fast.splitlines() if l.strip().startswith('|'))
pipes_4llm = sum(1 for l in md_4llm.splitlines() if l.strip().startswith('|'))
print(f'\nPipe-table lines: pymupdf_fast={pipes_fast}, pymupdf4llm={pipes_4llm}')

# Check key values
print(f'\n{"Keyword":<15} {"pymupdf_fast":<15} {"pymupdf4llm":<15}')
print('-' * 45)
for kw in [q['expected_keywords'][0] for q in questions]:
    in_fast = 'FOUND' if kw in md_fast else 'MISSING'
    in_4llm = 'FOUND' if kw in md_4llm else 'MISSING'
    print(f'{kw:<15} {in_fast:<15} {in_4llm:<15}')

Document: hubspot-q4
pymupdf_fast:     59043 chars,  2891 lines
pymupdf4llm:      44743 chars,   376 lines

Pipe-table lines: pymupdf_fast=261, pymupdf4llm=95

Keyword         pymupdf_fast    pymupdf4llm    
---------------------------------------------
846.7           FOUND           FOUND          
22.6%           FOUND           FOUND          
288,706         FOUND           FOUND          
3,854,151       FOUND           FOUND          
709,045         FOUND           FOUND          
884,945         FOUND           FOUND          
3.69            FOUND           FOUND          
829.0           FOUND           FOUND          
1.04            FOUND           FOUND          
1.0 billion     FOUND           FOUND          


In [22]:
print(md_4llm)

**==> picture [75 x 21] intentionally omitted <==**

## **HubSpot Reports Strong Q4 and Full Year 2025 Results** 

February 11, 2026 

_Q4'25 revenue grew 20% on an as-reported basis and 18% in constant currency compared to Q4'24 Full year 2025 revenue grew 19% on an as-reported basis and 18% in constant currency compared to 2024_ 

CAMBRIDGE, Mass.--(BUSINESS WIRE)--Feb. 11, 2026-- HubSpot, Inc. (NYSE: HUBS), the customer platform for scaling companies, today announced financial results for the fourth quarter and full year ended December 31, 2025. 

## **Financial Highlights:** 

## **Revenue** 

_Fourth Quarter 2025:_ 

- Total revenue was $846.7 million, up 20% on an as-reported basis and 18% in constant currency compared to Q4'24. 

   - Subscription revenue was $829.0 million, up 21% on an as-reported basis compared to Q4'24. 

   - Professional services and other revenue was $17.8 million, up 12% on an as-reported basis compared to Q4'24. 

_Full Year 2025:_ 

- Total revenue was

In [24]:
import pandas as pd

TOP_K = 5
RETRIEVAL = 'hybrid'  # 'dense', 'bm25', or 'hybrid'

def get_hit(data, q_emb, q, top_k, retrieval):
    dense_scores = cosine_similarity(q_emb, data['embeddings'])
    bm25_scores = data['bm25'].score(q['question'])
    if retrieval == 'dense': scores = dense_scores
    elif retrieval == 'bm25': scores = bm25_scores
    else: scores = hybrid_scores(dense_scores, bm25_scores)
    top_idx = np.argsort(scores)[::-1][:top_k]
    top_text = ' '.join(data['texts'][i] for i in top_idx)
    return any(kw.lower() in top_text.lower() for kw in q['expected_keywords'])

names = list(chunker_configs.keys())

# Collect results
rows = []
for q in questions:
    q_emb = embedder.embed_query(q['question'])
    for n in names:
        fh = get_hit(cd_fast[n], q_emb, q, TOP_K, RETRIEVAL)
        lh = get_hit(cd_4llm[n], q_emb, q, TOP_K, RETRIEVAL)
        rows.append({
            'question_id': q['id'],
            'question': q['question'],
            'expected_keywords': ', '.join(q['expected_keywords']),
            'chunker': n,
            'pymupdf_fast': 'HIT' if fh else 'MISS',
            'pymupdf4llm': 'HIT' if lh else 'MISS',
            'changed': '' if fh == lh else ('improved' if lh else 'regressed'),
        })

df = pd.DataFrame(rows)

# Save to CSV
out_path = PROJECT_ROOT / 'experiments' / 'chunking' / 'results' / f'debug_{DOC}.csv'
df.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

# Also make a pivot view: questions as rows, chunkers as columns, parser as sub-columns
for parser_col, label in [('pymupdf_fast', 'pymupdf_fast'), ('pymupdf4llm', 'pymupdf4llm')]:
    pivot = df.pivot(index=['question_id', 'question'], columns='chunker', values=parser_col)
    pivot_path = PROJECT_ROOT / 'experiments' / 'chunking' / 'results' / f'debug_{DOC}_{label}_pivot.csv'
    pivot.to_csv(pivot_path)
    print(f'Saved pivot to {pivot_path}')

# Show the differences
diffs = df[df['changed'] != '']
if len(diffs) > 0:
    print(f'\n{len(diffs)} differences found:')
    display(diffs[['question_id', 'question', 'chunker', 'pymupdf_fast', 'pymupdf4llm', 'changed']])
else:
    print('\nNo differences — both parsers produce identical results')

# Show summary
print(f'\nSummary (top-{TOP_K}, {RETRIEVAL}):')
summary = df.groupby('chunker').agg(
    fast_hits=('pymupdf_fast', lambda x: (x == 'HIT').sum()),
    llm_hits=('pymupdf4llm', lambda x: (x == 'HIT').sum()),
).assign(
    total=len(questions),
    fast_pct=lambda x: (x.fast_hits / len(questions) * 100).round(0).astype(int),
    llm_pct=lambda x: (x.llm_hits / len(questions) * 100).round(0).astype(int),
)
display(summary)

Saved to /Users/sarath/work/doc-parser/experiments/chunking/results/debug_hubspot-q4.csv
Saved pivot to /Users/sarath/work/doc-parser/experiments/chunking/results/debug_hubspot-q4_pymupdf_fast_pivot.csv
Saved pivot to /Users/sarath/work/doc-parser/experiments/chunking/results/debug_hubspot-q4_pymupdf4llm_pivot.csv

4 differences found:


,question_id,question,chunker,pymupdf_fast,pymupdf4llm,changed
18,hub_04,What was HubSpot's total assets as of December...,page_level,MISS,HIT,improved
21,hub_04,What was HubSpot's total assets as of December...,structure_aware,HIT,MISS,regressed
22,hub_04,What was HubSpot's total assets as of December...,table_aware,HIT,MISS,regressed
24,hub_05,What was HubSpot's gross profit in Q4 2025?,page_level,MISS,HIT,improved



Summary (top-5, hybrid):


,fast_hits,llm_hits,total,fast_pct,llm_pct
chunker,,,,,
element_type,8,8,10,80,80
fixed_512_overlap,8,8,10,80,80
heading_based,8,8,10,80,80
page_level,8,10,10,80,100
structure_aware,9,8,10,90,80
table_aware,9,8,10,90,80


In [13]:
# Find and display the first pipe-table block from pymupdf4llm
lines = md_4llm.split('\n')
in_table = False
table_start = None

for i, line in enumerate(lines):
    if line.strip().startswith('|') and not in_table:
        table_start = i
        in_table = True
    elif not line.strip().startswith('|') and in_table:
        print(f'First table: lines {table_start}-{i} ({i - table_start} lines)')
        print()
        for j in range(table_start, min(i, table_start + 20)):
            print(lines[j][:150])
        if i - table_start > 20:
            print(f'... ({i - table_start} lines total)')
        break

First table: lines 144-153 (9 lines)

|Current assets:<br>Cash and cash equivalents<br> <br>Short-term investments<br>Accounts receivable<br>Deferred commission expense<br>Prepaid expenses
|---|---|---|
|||<br>2,633,603<br> <br>154,212<br> <br>114,165<br> <br>154,484<br> <br>216,230<br> <br>160,814<br> <br>115,254<br> <br>37,563<br> <br>209,508|
|||<br>3,795,833|
|||<br>3,649<br> <br>67,442<br> <br>102,043<br> <br>125,135<br> <br>32,693<br> <br>458,184<br> <br>784,253|
|||<br>1,573,399<br> <br>254,539<br> <br>3,969<br> <br>55,640|
|||<br>1,887,547<br> <br>52<br> <br>—<br> <br>2,713,697<br> <br>(5,654)<br> <br>(799,809)|
|||<br>1,908,286|
|||$ 3,795,833|


## 3. Chunk both outputs and compare

In [14]:
# Chunkers to compare
chunker_configs = {
    'page_level': PageLevelChunker(),
    'fixed_512_overlap': FixedSizeTokenChunker(chunk_size=512, overlap=102),
    'element_type': ElementTypeChunker(),
    'structure_aware': StructureAwareChunker(),
    'table_aware': TableAwareChunker(),
    'heading_based': HeadingBasedChunker(),
}

def build_chunker_data(markdown, label):
    data = {}
    for name, chunker in chunker_configs.items():
        chunks = chunker.chunk(markdown)
        texts = [c.text for c in chunks]
        embeddings = embedder.embed_texts(texts)
        bm25 = BM25Index(texts)
        table_count = sum(1 for c in chunks if c.metadata.get('chunk_type') == ChunkType.TABLE)
        data[name] = {'chunks': chunks, 'texts': texts, 'embeddings': embeddings, 'bm25': bm25}
        print(f'  {name}: {len(chunks)} chunks (tables: {table_count}), avg {np.mean([count_tokens(t) for t in texts]):.0f} tok')
    return data

print('pymupdf_fast:')
cd_fast = build_chunker_data(md_fast, 'fast')
print()
print('pymupdf4llm:')
cd_4llm = build_chunker_data(md_4llm, '4llm')

pymupdf_fast:
  page_level: 10 chunks (tables: 0), avg 1832 tok
  fixed_512_overlap: 49 chunks (tables: 0), avg 434 tok
  element_type: 372 chunks (tables: 181), avg 49 tok
  structure_aware: 379 chunks (tables: 181), avg 51 tok
  table_aware: 390 chunks (tables: 181), avg 47 tok
  heading_based: 27 chunks (tables: 8), avg 677 tok

pymupdf4llm:
  page_level: 1 chunks (tables: 0), avg 12885 tok
  fixed_512_overlap: 31 chunks (tables: 0), avg 468 tok
  element_type: 28 chunks (tables: 15), avg 459 tok
  structure_aware: 46 chunks (tables: 15), avg 284 tok
  table_aware: 38 chunks (tables: 15), avg 337 tok
  heading_based: 31 chunks (tables: 8), avg 415 tok


## 4. Side-by-side HIT/MISS matrix

In [23]:
TOP_K = 5
RETRIEVAL = 'hybrid'  # 'dense', 'bm25', or 'hybrid'

def get_hit(data, q_emb, q, top_k, retrieval):
    dense_scores = cosine_similarity(q_emb, data['embeddings'])
    bm25_scores = data['bm25'].score(q['question'])
    if retrieval == 'dense': scores = dense_scores
    elif retrieval == 'bm25': scores = bm25_scores
    else: scores = hybrid_scores(dense_scores, bm25_scores)
    top_idx = np.argsort(scores)[::-1][:top_k]
    top_text = ' '.join(data['texts'][i] for i in top_idx)
    return any(kw.lower() in top_text.lower() for kw in q['expected_keywords'])

names = list(chunker_configs.keys())
header = f"{'Q_ID':<10} {'Question':<40}"
for n in names:
    header += f" fast_{n[:8]:>10}  4llm_{n[:8]:>10}"
print(header)
print('-' * len(header))

# Track totals
fast_hits = {n: 0 for n in names}
llm_hits = {n: 0 for n in names}

for q in questions:
    q_emb = embedder.embed_query(q['question'])
    row = f"{q['id']:<10} {q['question'][:38]:<40}"
    for n in names:
        fh = get_hit(cd_fast[n], q_emb, q, TOP_K, RETRIEVAL)
        lh = get_hit(cd_4llm[n], q_emb, q, TOP_K, RETRIEVAL)
        fast_hits[n] += fh
        llm_hits[n] += lh
        fs = '  HIT' if fh else ' MISS'
        ls = '  HIT' if lh else ' MISS'
        row += f' {fs:>10}  {ls:>10}'
    print(row)

# Totals
print('-' * len(header))
total_row = f"{'TOTAL':<10} {'':40}"
for n in names:
    total_row += f" {fast_hits[n]:>10}  {llm_hits[n]:>10}"
print(total_row)

Q_ID       Question                                 fast_  page_lev  4llm_  page_lev fast_  fixed_51  4llm_  fixed_51 fast_  element_  4llm_  element_ fast_  structur  4llm_  structur fast_  table_aw  4llm_  table_aw fast_  heading_  4llm_  heading_
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
hub_01     What was HubSpot's total revenue in Q4          HIT         HIT        HIT         HIT        HIT         HIT        HIT         HIT        HIT         HIT        HIT         HIT
hub_02     What was HubSpot's non-GAAP operating           HIT         HIT        HIT         HIT        HIT         HIT        HIT         HIT        HIT         HIT        HIT         HIT
hub_03     How many customers did HubSpot have as          HIT         HIT        HIT         HIT        HIT         HIT

## 5. Deep-dive: inspect a specific question

In [16]:
# ---- EDIT THIS ----
Q_ID = 'hub_04'
CHUNKER = 'table_aware'
# -------------------

q = next(q for q in questions if q['id'] == Q_ID)

for label, cd in [('pymupdf_fast', cd_fast), ('pymupdf4llm', cd_4llm)]:
    data = cd[CHUNKER]
    chunks = data['chunks']
    texts = data['texts']

    print(f"\n{'='*70}")
    print(f"{label} + {CHUNKER} ({len(chunks)} chunks)")
    print(f"Question: {q['question']}")
    print(f"Expected: {q['expected_keywords']}")
    print(f"{'='*70}")

    # Find answer chunks
    answer_chunks = [i for i, t in enumerate(texts)
                     if any(kw.lower() in t.lower() for kw in q['expected_keywords'])]

    if not answer_chunks:
        print('!! ANSWER NOT IN ANY CHUNK')
        continue

    print(f'Answer found in chunk(s): {answer_chunks}')
    for i in answer_chunks:
        page = chunks[i].metadata.get('page_num', '?')
        ctype = chunks[i].metadata.get('chunk_type', '?')
        tok = count_tokens(texts[i])
        print(f"\n  Chunk {i} [page {page}, {ctype}, {tok} tok]:")
        print(f"  {texts[i][:400]}...")

    # Rankings
    q_emb = embedder.embed_query(q['question'])
    dense_scores = cosine_similarity(q_emb, data['embeddings'])
    bm25_scores = data['bm25'].score(q['question'])
    hyb = hybrid_scores(dense_scores, bm25_scores)

    dense_ranks = np.argsort(np.argsort(-dense_scores))
    bm25_ranks = np.argsort(np.argsort(-bm25_scores))
    hybrid_ranks = np.argsort(np.argsort(-hyb))

    print(f"\n  {'Chunk':<8} {'Dense':<10} {'BM25':<10} {'Hybrid':<10}")
    for i in answer_chunks:
        print(f"  {i:<8} #{dense_ranks[i]+1:<9} #{bm25_ranks[i]+1:<9} #{hybrid_ranks[i]+1:<9}")


pymupdf_fast + table_aware (390 chunks)
Question: What was HubSpot's total assets as of December 31, 2025?
Expected: ['3,854,151']
Answer found in chunk(s): [8]

  Chunk 8 [page 4, ChunkType.TEXT, 512 tok]:
  Current assets:
 
   
  
Cash and cash equivalents
 $
882,242  $
512,667 
Short-term investments
  
821,552   
1,556,828 
Accounts receivable
  
419,146   
334,829 
Deferred commission expense
  
226,184   
148,693 
Prepaid expenses and other current assets
  
100,611   
80,586 
Total current assets
  
2,449,735   
2,633,603 
Long-term investments
  
136,662   
154,212 
Property and equipment, net...

  Chunk    Dense      BM25       Hybrid    
  8        #13        #5         #4        

pymupdf4llm + table_aware (38 chunks)
Question: What was HubSpot's total assets as of December 31, 2025?
Expected: ['3,854,151']
Answer found in chunk(s): [7]

  Chunk 7 [page 1, ChunkType.TABLE, 764 tok]:
  |Current assets:<br>Cash and cash equivalents<br> <br>Short-term investments<br>Accounts

## 6. Compare chunk boundaries for a keyword

In [18]:
# ---- EDIT THIS ----
KEYWORD = '3,854,151'
CHUNKER = 'table_aware'
# -------------------

for label, cd in [('pymupdf_fast', cd_fast), ('pymupdf4llm', cd_4llm)]:
    data = cd[CHUNKER]
    print(f"\n{'='*70}")
    print(f"{label} + {CHUNKER} — {len(data['chunks'])} chunks")
    print(f"{'='*70}")
    found = False
    for i, (chunk, text) in enumerate(zip(data['chunks'], data['texts'])):
        if KEYWORD.lower() in text.lower():
            found = True
            page = chunk.metadata.get('page_num', '?')
            ctype = chunk.metadata.get('chunk_type', '?')
            tok = count_tokens(text)
            pos = text.lower().find(KEYWORD.lower())
            start = max(0, pos - 100)
            end = min(len(text), pos + len(KEYWORD) + 100)
            # context = text[start:end].replace('\n', ' ')
            context = text.replace('\n', ' ')
            print(f"\n  Chunk {i} [page {page}, {ctype}, {tok} tok]")
            print(f"  ...{context}...")
    if not found:
        print(f"  '{KEYWORD}' NOT FOUND in any chunk")


pymupdf_fast + table_aware — 390 chunks

  Chunk 8 [page 4, ChunkType.TEXT, 512 tok]
  ...Current assets:          Cash and cash equivalents  $ 882,242  $ 512,667  Short-term investments    821,552    1,556,828  Accounts receivable    419,146    334,829  Deferred commission expense    226,184    148,693  Prepaid expenses and other current assets    100,611    80,586  Total current assets    2,449,735    2,633,603  Long-term investments    136,662    154,212  Property and equipment, net    141,869    114,165  Capitalized software development costs, net    213,794    154,484  Right-of-use assets    200,821    216,230  Deferred commission expense, net of current portion    218,991    160,814  Other assets    165,602    115,254  Intangible assets, net    35,225    37,563  Goodwill    291,452    209,508  Total assets    3,854,151    3,795,833  Liabilities and stockholders’ equity          Current liabilities:          Accounts payable    24,764    3,649  Accrued compensation costs    99,19

## 7. Ask a custom question

In [17]:
# ---- EDIT THIS ----
QUESTION = "What was HubSpot's total assets as of December 31, 2025?"
EXPECTED = ['3,854,151']
TOP_K = 5
RETRIEVAL = 'hybrid'
USE_4LLM = True  # False = pymupdf_fast, True = pymupdf4llm
# -------------------

cd = cd_4llm if USE_4LLM else cd_fast
label = 'pymupdf4llm' if USE_4LLM else 'pymupdf_fast'
q_emb = embedder.embed_query(QUESTION)

for name, data in cd.items():
    chunks = data['chunks']
    texts = data['texts']
    dense_scores = cosine_similarity(q_emb, data['embeddings'])
    bm25_scores = data['bm25'].score(QUESTION)
    if RETRIEVAL == 'dense': scores = dense_scores
    elif RETRIEVAL == 'bm25': scores = bm25_scores
    else: scores = hybrid_scores(dense_scores, bm25_scores)

    top_idx = np.argsort(scores)[::-1][:TOP_K]
    top_text = ' '.join(texts[i] for i in top_idx)
    hit = any(kw.lower() in top_text.lower() for kw in EXPECTED)

    print(f"\n{'='*80}")
    print(f"{label} + {name} — {'HIT' if hit else 'MISS'} (top-{TOP_K}, {RETRIEVAL})")
    print(f"{'='*80}")

    for rank, idx in enumerate(top_idx):
        page = chunks[idx].metadata.get('page_num', '?')
        ctype = chunks[idx].metadata.get('chunk_type', '?')
        tok = count_tokens(texts[idx])
        has_answer = any(kw.lower() in texts[idx].lower() for kw in EXPECTED)
        marker = ' << ANSWER' if has_answer else ''
        preview = texts[idx][:200].replace('\n', ' ')
        print(f"\n  #{rank+1} [page {page}, {ctype}, {tok} tok] dense={dense_scores[idx]:.3f} bm25={bm25_scores[idx]:.1f}{marker}")
        print(f"  {preview}...")


pymupdf4llm + page_level — HIT (top-5, hybrid)

  #1 [page 1, ChunkType.MIXED, 12885 tok] dense=0.431 bm25=-4.8 << ANSWER
  **==> picture [75 x 21] intentionally omitted <==**  ## **HubSpot Reports Strong Q4 and Full Year 2025 Results**   February 11, 2026   _Q4'25 revenue grew 20% on an as-reported basis and 18% in consta...

pymupdf4llm + fixed_512_overlap — MISS (top-5, hybrid)

  #1 [page 1, ChunkType.MIXED, 507 tok] dense=0.595 bm25=6.4
  **==> picture [75 x 21] intentionally omitted <==** ## **HubSpot Reports Strong Q4 and Full Year 2025 Results**  February 11, 2026  _Q4'25 revenue grew 20% on an as-reported basis and 18% in constant ...

  #2 [page 1, ChunkType.MIXED, 485 tok] dense=0.600 bm25=5.5
  ## **Additional Recent Business Highlights**  - Grew Customers to 288,706 at December 31, 2025, up 16% from December 31, 2024. Average Subscription Revenue Per Customer was $11,683 during the fourth q...

  #3 [page 1, ChunkType.MIXED, 348 tok] dense=0.593 bm25=4.1
  ||||||$ 548,237

## 8. Dump all chunks for a chunker + question (both parsers)

In [25]:
# ---- EDIT THIS ----
Q_ID = 'hub_04'
CHUNKER = 'table_aware'
RETRIEVAL = 'hybrid'  # 'dense', 'bm25', or 'hybrid'
# -------------------

q = next(q for q in questions if q['id'] == Q_ID)
q_emb = embedder.embed_query(q['question'])

output = {
    'question_id': q['id'],
    'question': q['question'],
    'expected_keywords': q['expected_keywords'],
    'chunker': CHUNKER,
    'retrieval': RETRIEVAL,
    'parsers': {}
}

for label, cd in [('pymupdf_fast', cd_fast), ('pymupdf4llm', cd_4llm)]:
    data = cd[CHUNKER]
    chunks = data['chunks']
    texts = data['texts']

    dense_scores = cosine_similarity(q_emb, data['embeddings'])
    bm25_scores = data['bm25'].score(q['question'])
    if RETRIEVAL == 'dense': scores = dense_scores
    elif RETRIEVAL == 'bm25': scores = bm25_scores
    else: scores = hybrid_scores(dense_scores, bm25_scores)

    ranked_idx = np.argsort(scores)[::-1]

    chunk_list = []
    for rank, idx in enumerate(ranked_idx):
        has_answer = any(kw.lower() in texts[idx].lower() for kw in q['expected_keywords'])
        chunk_list.append({
            'rank': rank + 1,
            'chunk_index': int(idx),
            'page_num': chunks[idx].metadata.get('page_num', None),
            'chunk_type': str(chunks[idx].metadata.get('chunk_type', '')),
            'tokens': count_tokens(texts[idx]),
            'dense_score': round(float(dense_scores[idx]), 4),
            'bm25_score': round(float(bm25_scores[idx]), 2),
            'hybrid_score': round(float(scores[idx]), 6),
            'has_answer': has_answer,
            'text': texts[idx],
        })

    output['parsers'][label] = {
        'total_chunks': len(chunks),
        'answer_found_in_chunks': [c['chunk_index'] for c in chunk_list if c['has_answer']],
        'chunks': chunk_list,
    }

# Save to JSON
out_path = PROJECT_ROOT / 'experiments' / 'chunking' / 'results' / f'chunks_{DOC}_{CHUNKER}_{Q_ID}.json'
with open(out_path, 'w') as f:
    json.dump(output, f, indent=2)
print(f'Saved all chunks to: {out_path}')

# Print summary
for label in ['pymupdf_fast', 'pymupdf4llm']:
    p = output['parsers'][label]
    answer_in = p['answer_found_in_chunks']
    answer_ranks = [c['rank'] for c in p['chunks'] if c['has_answer']]
    print(f'\n{label}: {p["total_chunks"]} chunks')
    if answer_in:
        print(f'  Answer in chunk(s): {answer_in}, ranked at: {answer_ranks}')
    else:
        print(f'  Answer NOT FOUND in any chunk')

Saved all chunks to: /Users/sarath/work/doc-parser/experiments/chunking/results/chunks_hubspot-q4_table_aware_hub_04.json

pymupdf_fast: 390 chunks
  Answer in chunk(s): [8], ranked at: [4]

pymupdf4llm: 38 chunks
  Answer in chunk(s): [7], ranked at: [19]


## 9. Chonkie TableChunker experiment

Use [Chonkie's TableChunker](https://docs.chonkie.ai/oss/chunkers/table-chunker) to split markdown tables by rows (preserving headers).
Compare against our custom `TableAwareChunker` for retrieval quality.

In [ ]:
try:
    from chonkie import TableChunker
    print('chonkie imported OK')
except ImportError:
    print('Installing chonkie...')
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'chonkie'])
    from chonkie import TableChunker
    print('chonkie installed and imported OK')

In [27]:
import re

def extract_markdown_tables(markdown):
    """Extract pipe-delimited markdown tables from text, returning (table_str, start_line, end_line) tuples."""
    lines = markdown.split('\n')
    tables = []
    table_lines = []
    table_start = None

    for i, line in enumerate(lines):
        if line.strip().startswith('|'):
            if table_start is None:
                table_start = i
            table_lines.append(line)
        else:
            if table_lines:
                tables.append(('\n'.join(table_lines), table_start, i))
                table_lines = []
                table_start = None
    if table_lines:
        tables.append(('\n'.join(table_lines), table_start, len(lines)))

    return tables

def extract_non_table_text(markdown):
    """Extract non-table text segments from markdown."""
    lines = markdown.split('\n')
    segments = []
    current = []

    for line in lines:
        if line.strip().startswith('|'):
            if current:
                text = '\n'.join(current).strip()
                if text:
                    segments.append(text)
                current = []
        else:
            current.append(line)
    if current:
        text = '\n'.join(current).strip()
        if text:
            segments.append(text)

    return segments


# ---- EDIT THIS ----
CHONKIE_CHUNK_SIZE = 512  # max tokens per table chunk
# -------------------

# Use token-based chunking so tables split at a consistent token budget
# regardless of row count
chonkie_chunker = TableChunker(tokenizer='o200k_base', chunk_size=CHONKIE_CHUNK_SIZE)

def chonkie_chunk_markdown(markdown):
    """Chunk markdown: use Chonkie TableChunker for tables, fixed-size for text."""
    tables = extract_markdown_tables(markdown)
    text_segments = extract_non_table_text(markdown)

    all_chunks = []

    # Chunk tables with Chonkie (token-based, headers preserved automatically)
    for table_str, start, end in tables:
        try:
            chunks = chonkie_chunker.chunk(table_str)
            for c in chunks:
                all_chunks.append(Chunk(
                    text=c.text,
                    metadata={'chunk_type': ChunkType.TABLE, 'page_num': 1, 'source': 'chonkie_table'}
                ))
        except Exception as e:
            # Fallback: keep table as single chunk
            all_chunks.append(Chunk(
                text=table_str,
                metadata={'chunk_type': ChunkType.TABLE, 'page_num': 1, 'source': 'chonkie_table_fallback'}
            ))

    # Chunk text segments with fixed-size
    text_chunker = FixedSizeTokenChunker(chunk_size=512, overlap=102)
    full_text = '\n\n'.join(text_segments)
    text_chunks = text_chunker.chunk(full_text)
    for c in text_chunks:
        c.metadata['source'] = 'text_fixed'
    all_chunks.extend(text_chunks)

    return all_chunks


# Build data for both parsers
print(f'Chonkie TableChunker config: tokenizer=o200k_base, chunk_size={CHONKIE_CHUNK_SIZE} tokens\n')

cd_chonkie = {}
for label, md in [('pymupdf_fast', md_fast), ('pymupdf4llm', md_4llm)]:
    chunks = chonkie_chunk_markdown(md)
    texts = [c.text for c in chunks]
    embeddings = embedder.embed_texts(texts)
    bm25 = BM25Index(texts)
    table_count = sum(1 for c in chunks if c.metadata.get('chunk_type') == ChunkType.TABLE)
    cd_chonkie[label] = {'chunks': chunks, 'texts': texts, 'embeddings': embeddings, 'bm25': bm25}
    print(f'{label}: {len(chunks)} chunks (tables: {table_count}), avg {np.mean([count_tokens(t) for t in texts]):.0f} tok')

2026-03-20 07:51:33,862 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:33,864 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:33,864 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:33,864 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:33,864 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:33,865 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:33,865 | WARNING  | chonkie.chunker.table:chunk:128 - Table

Chonkie TableChunker config: tokenizer=o200k_base, chunk_size=512 tokens



2026-03-20 07:51:35,102 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:35,104 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.
2026-03-20 07:51:35,105 | WARNING  | chonkie.chunker.table:chunk:128 - Table must have at least a header, separator, and one data row. Skipping chunking.


pymupdf_fast: 66 chunks (tables: 28), avg 271 tok
pymupdf4llm: 45 chunks (tables: 31), avg 562 tok


In [28]:
# Compare Chonkie vs our TableAwareChunker on retrieval

TOP_K = 5
RETRIEVAL = 'hybrid'

print(f"{'Q_ID':<10} {'Question':<40} {'fast_ours':>10} {'fast_chonk':>10} {'4llm_ours':>10} {'4llm_chonk':>10}")
print('-' * 120)

totals = {'fast_ours': 0, 'fast_chonkie': 0, '4llm_ours': 0, '4llm_chonkie': 0}

for q in questions:
    q_emb = embedder.embed_query(q['question'])

    # Our table_aware chunker
    fh_ours = get_hit(cd_fast['table_aware'], q_emb, q, TOP_K, RETRIEVAL)
    lh_ours = get_hit(cd_4llm['table_aware'], q_emb, q, TOP_K, RETRIEVAL)

    # Chonkie
    fh_chonkie = get_hit(cd_chonkie['pymupdf_fast'], q_emb, q, TOP_K, RETRIEVAL)
    lh_chonkie = get_hit(cd_chonkie['pymupdf4llm'], q_emb, q, TOP_K, RETRIEVAL)

    totals['fast_ours'] += fh_ours
    totals['fast_chonkie'] += fh_chonkie
    totals['4llm_ours'] += lh_ours
    totals['4llm_chonkie'] += lh_chonkie

    def fmt(hit): return '  HIT' if hit else ' MISS'
    print(f"{q['id']:<10} {q['question'][:38]:<40} {fmt(fh_ours):>10} {fmt(fh_chonkie):>10} {fmt(lh_ours):>10} {fmt(lh_chonkie):>10}")

print('-' * 120)
print(f"{'TOTAL':<10} {'':40} {totals['fast_ours']:>10} {totals['fast_chonkie']:>10} {totals['4llm_ours']:>10} {totals['4llm_chonkie']:>10}")
print(f"\nOurs (table_aware):  fast={totals['fast_ours']}/{len(questions)}, 4llm={totals['4llm_ours']}/{len(questions)}")
print(f"Chonkie TableChunker: fast={totals['fast_chonkie']}/{len(questions)}, 4llm={totals['4llm_chonkie']}/{len(questions)}")

Q_ID       Question                                  fast_ours fast_chonk  4llm_ours 4llm_chonk
------------------------------------------------------------------------------------------------------------------------
hub_01     What was HubSpot's total revenue in Q4          HIT        HIT        HIT        HIT
hub_02     What was HubSpot's non-GAAP operating           HIT        HIT        HIT        HIT
hub_03     How many customers did HubSpot have as          HIT        HIT        HIT        HIT
hub_04     What was HubSpot's total assets as of           HIT        HIT       MISS       MISS
hub_05     What was HubSpot's gross profit in Q4          MISS       MISS       MISS       MISS
hub_06     What was the cash and cash equivalents          HIT        HIT        HIT        HIT
hub_07     What is the expected total revenue for          HIT        HIT        HIT        HIT
hub_08     What was HubSpot's subscription revenu          HIT        HIT        HIT        HIT
hub_09     What

In [29]:
# Inspect Chonkie table chunks — see what the chunker produces

# ---- EDIT THIS ----
PARSER = 'pymupdf4llm'  # 'pymupdf_fast' or 'pymupdf4llm'
KEYWORD = '3,854,151'    # keyword to highlight
# -------------------

data = cd_chonkie[PARSER]
chunks = data['chunks']
texts = data['texts']

table_chunks = [(i, c, t) for i, (c, t) in enumerate(zip(chunks, texts))
                if c.metadata.get('chunk_type') == ChunkType.TABLE]

print(f'{PARSER} — Chonkie table chunks: {len(table_chunks)} / {len(chunks)} total\n')

for i, chunk, text in table_chunks:
    tok = count_tokens(text)
    src = chunk.metadata.get('source', '?')
    has_kw = KEYWORD.lower() in text.lower() if KEYWORD else False
    marker = ' << KEYWORD' if has_kw else ''
    preview = text[:300].replace('\n', ' ')
    print(f'  Chunk {i} [{src}, {tok} tok]{marker}')
    print(f'  {preview}...\n')

pymupdf4llm — Chonkie table chunks: 31 / 45 total

  Chunk 0 [chonkie_table, 589 tok] << KEYWORD
  |Current assets:<br>Cash and cash equivalents<br> <br>Short-term investments<br>Accounts receivable<br>Deferred commission expense<br>Prepaid expenses and other current assets<br> <br>Total current assets<br>Long-term investments<br>Property and equipment, net<br>Capitalized software development cos...

  Chunk 1 [chonkie_table, 525 tok] << KEYWORD
  |Current assets:<br>Cash and cash equivalents<br> <br>Short-term investments<br>Accounts receivable<br>Deferred commission expense<br>Prepaid expenses and other current assets<br> <br>Total current assets<br>Long-term investments<br>Property and equipment, net<br>Capitalized software development cos...

  Chunk 2 [chonkie_table, 571 tok] << KEYWORD
  |Current assets:<br>Cash and cash equivalents<br> <br>Short-term investments<br>Accounts receivable<br>Deferred commission expense<br>Prepaid expenses and other current assets<br> <br>Total curre

In [31]:
# Deep-dive: Chonkie retrieval for a specific question

# ---- EDIT THIS ----
Q_ID = 'hub_04'
TOP_K = 10
RETRIEVAL = 'hybrid'
# -------------------

q = next(q for q in questions if q['id'] == Q_ID)
q_emb = embedder.embed_query(q['question'])

for label in ['pymupdf_fast', 'pymupdf4llm']:
    data = cd_chonkie[label]
    chunks = data['chunks']
    texts = data['texts']

    dense_scores = cosine_similarity(q_emb, data['embeddings'])
    bm25_scores = data['bm25'].score(q['question'])
    scores = hybrid_scores(dense_scores, bm25_scores)

    top_idx = np.argsort(scores)[::-1][:TOP_K]
    top_text = ' '.join(texts[i] for i in top_idx)
    hit = any(kw.lower() in top_text.lower() for kw in q['expected_keywords'])

    # Find which chunk has the answer
    answer_chunks = [i for i, t in enumerate(texts)
                     if any(kw.lower() in t.lower() for kw in q['expected_keywords'])]
    answer_ranks = [np.where(np.argsort(scores)[::-1] == i)[0][0] + 1 for i in answer_chunks]

    print(f"\n{'='*80}")
    print(f"Chonkie + {label} — {'HIT' if hit else 'MISS'} (top-{TOP_K}, {RETRIEVAL})")
    print(f"Q: {q['question']}")
    print(f"Expected: {q['expected_keywords']}")
    if answer_chunks:
        print(f"Answer in chunk(s) {answer_chunks}, ranked at {answer_ranks}")
    else:
        print("Answer NOT FOUND in any chunk")
    print(f"{'='*80}")

    for rank, idx in enumerate(top_idx):
        ctype = chunks[idx].metadata.get('chunk_type', '?')
        src = chunks[idx].metadata.get('source', '?')
        tok = count_tokens(texts[idx])
        has_answer = any(kw.lower() in texts[idx].lower() for kw in q['expected_keywords'])
        marker = ' << ANSWER' if has_answer else ''
        preview = texts[idx].replace('\n', ' ')
        print(f"\n  #{rank+1} [{ctype}, {src}, {tok} tok] dense={dense_scores[idx]:.3f} bm25={bm25_scores[idx]:.1f}{marker}")
        print(f"  {preview}...")


Chonkie + pymupdf_fast — HIT (top-10, hybrid)
Q: What was HubSpot's total assets as of December 31, 2025?
Expected: ['3,854,151']
Answer in chunk(s) [36, 37], ranked at [np.int64(5), np.int64(7)]

  #1 [ChunkType.MIXED, text_fixed, 493 tok] dense=0.601 bm25=8.4
  HubSpot Reports Strong Q4 and Full Year 2025 Results February 11, 2026 Q4'25 revenue grew 20% on an as-reported basis and 18% in constant currency compared to Q4'24 Full year 2025 revenue grew 19% on an as-reported basis and 18% in constant currency compared to 2024 CAMBRIDGE, Mass.--(BUSINESS WIRE)--Feb. 11, 2026-- HubSpot, Inc. (NYSE: HUBS), the customer platform for scaling companies, today announced financial results for the fourth quarter and full year ended December 31, 2025. Financial Highlights: Revenue Fourth Quarter 2025: Total revenue was $846.7 million, up 20% on an as-reported basis and 18% in constant currency compared to Q4'24. Subscription revenue was $829.0 million, up 21% on an as-reported basis compared to 

In [ ]:
# Sweep Chonkie chunk_size (tokens) to find optimal table chunk size

TOP_K = 5
RETRIEVAL = 'hybrid'
TOKEN_SIZES = [128, 256, 512, 768, 1024]

print(f"Sweeping Chonkie TableChunker token sizes: {TOKEN_SIZES}")
print(f"{'tok_size':>8} {'parser':<15} {'chunks':>7} {'tables':>7} {'avg_tok':>8} {'hits':>5} {'total':>6}")
print('-' * 65)

for tok_size in TOKEN_SIZES:
    tc = TableChunker(tokenizer='o200k_base', chunk_size=tok_size)

    for label, md in [('pymupdf_fast', md_fast), ('pymupdf4llm', md_4llm)]:
        tables = extract_markdown_tables(md)
        text_segments = extract_non_table_text(md)
        all_chunks = []

        for table_str, start, end in tables:
            try:
                chunks = tc.chunk(table_str)
                for c in chunks:
                    all_chunks.append(Chunk(
                        text=c.text,
                        metadata={'chunk_type': ChunkType.TABLE, 'page_num': 1}
                    ))
            except:
                all_chunks.append(Chunk(
                    text=table_str,
                    metadata={'chunk_type': ChunkType.TABLE, 'page_num': 1}
                ))

        text_chunker = FixedSizeTokenChunker(chunk_size=512, overlap=102)
        full_text = '\n\n'.join(text_segments)
        all_chunks.extend(text_chunker.chunk(full_text))

        texts = [c.text for c in all_chunks]
        embeddings = embedder.embed_texts(texts)
        bm25 = BM25Index(texts)
        data = {'chunks': all_chunks, 'texts': texts, 'embeddings': embeddings, 'bm25': bm25}

        table_count = sum(1 for c in all_chunks if c.metadata.get('chunk_type') == ChunkType.TABLE)
        avg_tok = np.mean([count_tokens(t) for t in texts])

        hits = 0
        for q in questions:
            q_emb = embedder.embed_query(q['question'])
            if get_hit(data, q_emb, q, TOP_K, RETRIEVAL):
                hits += 1

        print(f"{tok_size:>8} {label:<15} {len(all_chunks):>7} {table_count:>7} {avg_tok:>8.0f} {hits:>5} {len(questions):>6}")